# 5장. 데이터 나누기

In [1]:
# [준비 셀] 모든 노트북의 첫 코드 셀입니다. 항상 먼저 실행하세요.
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parents[1]          # docs/manual → 프로젝트 루트
assert (ROOT / "src/bearing_kd").exists(), "docs/manual 폴더에서 노트북을 열어야 합니다"
CODE = ROOT / "docs/manual/code"                 # 챕터별 완성 스크립트 위치
sys.path.insert(0, str(CODE))                    # 스크립트의 함수를 import 해서 쓰기 위해
print("프로젝트 루트 확인 완료:", ROOT.name)

프로젝트 루트 확인 완료: vib-onnx


2장에서 train / validation / test를 나누는 이유를 봤습니다. 이 장에서는 **무엇을 단위로 나누느냐**를 다룹니다. 이 프로젝트에서 가장 비싼 교훈을 얻은 부분입니다.

## 5.1 규칙 1: 윈도우가 아니라 "녹음 파일" 단위로 나눈다

4장에서 윈도우끼리 절반씩 겹친다고 했습니다. 같은 파일의 윈도우 1번을 train에, 2번을 test에 넣으면 **test 윈도우의 절반이 train에 이미 있던 데이터**입니다. 시험 문제의 절반을 미리 알려준 셈입니다. 이를 **데이터 누수(data leakage)**라고 합니다.

그래서 한 파일에서 나온 윈도우들은 반드시 **같은 세트**로 보냅니다.

## 5.2 규칙 2: 새 베어링 성능을 알고 싶으면 "베어링" 단위로 나눈다

| 방식 | 나누는 단위 | test가 답하는 질문 |
|---|---|---|
| 파일 단위 | 같은 베어링의 다른 녹음 파일 | "**이미 본 베어링**의 다른 날 측정도 맞히나?" |
| **베어링 단위 (holdout)** | 베어링 개체 통째로 | "**처음 보는 베어링**도 맞히나?" |

현장에 설치할 모델은 당연히 처음 보는 베어링을 판정해야 합니다. 그런데 이 프로젝트의 첫 평가는 파일 단위였습니다.

| 이 프로젝트의 실제 결과 | 파일 단위 test | 베어링 단위 test |
|---|---|---|
| 정확도 | **99.6%** | **58–65%** |

같은 모델인데 나누는 방식만 바꿨더니 성능이 크게 떨어졌습니다. 모델이 고장의 특징보다 **각 베어링 개체의 버릇을 외웠기** 때문입니다. validation도 같은 문제가 있어서, 같은 베어링의 다른 파일로 만든 validation은 98–100%였습니다.

## 5.3 규칙 3: test에는 정상과 고장이 모두 있어야 한다

고장 베어링만으로 test를 만들면, 무조건 "고장"이라고 답하는 엉터리 모델도 100%가 나옵니다. 정상을 고장이라고 잘못 말하는 **오경보**를 잴 수 없기 때문입니다. 매뉴얼 코드와 프로젝트 코드는 이 경우 에러를 냅니다.

In [2]:
import json
from collections import Counter
from pathlib import Path
import ch05_split as ch05

PROC = ROOT / "outputs/manual/processed"
paths = sorted(PROC.rglob("*_windows.npy"))
print("전처리된 파일:", len(paths), dict(Counter(p.parent.name for p in paths)))

전처리된 파일: 32 {'K001': 8, 'K002': 8, 'KA03': 8, 'KA04': 8}


## 5.4 방식 1: 파일 단위 70/15/15

라벨별로 따로 섞은 뒤 70/15/15로 자릅니다. 그래야 각 세트의 정상/고장 비율이 비슷해집니다.

In [3]:
groups = ch05.create_split(paths, seed=42)
for name, files in groups.items():
    print(f"{name:10s} {len(files):2d}개", dict(sorted(Counter(Path(f).parent.name for f in files).items())))

train      22개 {'K001': 5, 'K002': 6, 'KA03': 6, 'KA04': 5}
validation  4개 {'K001': 1, 'K002': 1, 'KA03': 1, 'KA04': 1}
test        6개 {'K001': 2, 'K002': 1, 'KA03': 1, 'KA04': 2}


모든 베어링이 train과 test 양쪽에 들어가 있습니다. 이 test로는 새 베어링 성능을 알 수 없습니다.

## 5.5 방식 2: 베어링 단위 holdout

정상 K002와 고장 KA04를 통째로 test로 뺍니다. 나머지(K001, KA03)의 파일을 85/15로 train/validation에 나눕니다.

In [4]:
groups = ch05.create_split(paths, test_bearings=["K002", "KA04"], seed=42)
for name, files in groups.items():
    print(f"{name:10s} {len(files):2d}개", dict(sorted(Counter(Path(f).parent.name for f in files).items())))

train_bearings = {Path(f).parent.name for f in groups["train"] + groups["validation"]}
test_bearings = {Path(f).parent.name for f in groups["test"]}
print("\ntrain/validation 베어링:", train_bearings, " test 베어링:", test_bearings)
print("겹치는 베어링:", train_bearings & test_bearings or "없음")

train      12개 {'K001': 6, 'KA03': 6}
validation  4개 {'K001': 2, 'KA03': 2}
test       16개 {'K002': 8, 'KA04': 8}

train/validation 베어링: {'K001', 'KA03'}  test 베어링: {'K002', 'KA04'}
겹치는 베어링: 없음


In [5]:
try:
    ch05.create_split(paths, test_bearings=["KA04"])      # 고장 베어링만 test로
except ValueError as e:
    print("거부됨:", e)

거부됨: test 베어링에 정상과 고장이 모두 있어야 합니다 (한쪽만 있으면 오판율을 잴 수 없음)


## 5.6 seed: 같은 분할을 다시 만들기

`seed`는 무작위 섞기의 시작점입니다. seed가 같으면 몇 번을 돌려도 같은 분할이 나옵니다. 그래서 결과를 다른 사람이 **재현**할 수 있습니다.

In [6]:
a = ch05.create_split(paths, test_bearings=["K002", "KA04"], seed=42)
b = ch05.create_split(paths, test_bearings=["K002", "KA04"], seed=42)
c = ch05.create_split(paths, test_bearings=["K002", "KA04"], seed=7)
print("seed 42 두 번 같음:", a == b, "  seed 42 vs 7 같음:", a == c)

seed 42 두 번 같음: True   seed 42 vs 7 같음: False


## 5.7 split JSON 저장

분할 결과는 **파일 경로 목록**을 JSON으로 저장합니다. 경로는 프로젝트 루트 기준 상대 경로라서 다른 컴퓨터에서도 그대로 쓸 수 있습니다. 이 프로젝트의 실제 분할은 `data/splits/bearing_holdout_v3.json`입니다.

In [7]:
%run code/ch05_split.py --test-bearings K002 KA04

train      파일  12개  {'K001': 6, 'KA03': 6}
validation 파일   4개  {'K001': 2, 'KA03': 2}
test       파일  16개  {'K002': 8, 'KA04': 8}

저장: outputs/manual/split_holdout.json


In [8]:
split = json.loads((ROOT / "outputs/manual/split_holdout.json").read_text())
print({k: len(v) for k, v in split.items()})
print("train 첫 2개:", split["train"][:2])

{'train': 12, 'validation': 4, 'test': 16}
train 첫 2개: ['outputs/manual/processed/K001/N15_M07_F04_K001_1_windows.npy', 'outputs/manual/processed/KA03/N15_M07_F10_KA03_1_windows.npy']


실제 프로젝트에서는:
```bash
bearing-kd split --processed-dir data/processed/windows_v3 --test-bearings K004 KB24 KB27 \
    --output data/splits/bearing_holdout_v3.json
```

> **아직 남은 숙제:** validation도 베어링 단위로 나누는 것이 좋습니다. 지금은 validation이 train과 같은 베어링의 다른 파일이라, 학습 중 "가장 좋은 epoch"를 고를 때 실제 성능보다 좋아 보입니다.

## 5장 완성 코드: `code/ch05_split.py`

In [9]:
from IPython.display import Code
Code(filename=str(CODE / "ch05_split.py"), language="python")

"""5장 완성 코드: 전처리된 파일 목록을 train / validation / test로 나눈 split JSON을 만든다.

두 가지 방식 (src/bearing_kd/data.py의 create_split과 같은 규칙):
    1) 녹음 파일 단위 70/15/15  : 같은 베어링의 다른 파일이 train과 test에 함께 들어간다.
    2) 베어링 단위 holdout      : --test-bearings 로 지정한 베어링은 통째로 test.
                                  나머지 베어링의 파일을 85/15로 train/validation에 나눈다.

실행 (프로젝트 루트에서, 4장 맛보기 결과 사용):
    python docs/manual/code/ch05_split.py --test-bearings K002 KA04
    python docs/manual/code/ch05_split.py --output outputs/manual/split_file.json   # 파일 단위
"""

import argparse
import json
import random
import re
from collections import Counter
from pathlib import Path

ROOT = Path(__file__).resolve().parents[3]


def bearing_label(code):
    if re.fullmatch(r"K\d{3}", code):
        return 0
    if re.fullmatch(r"K[ABI]\d{2}", code):
        return 1
    raise ValueError(f"알 수 없는 베어링 코드: {code}")


def create_split(paths, test_bearings=None, seed=42):
    rng = random.Random(seed)            # seed가 같으면 언제나 같은 분할 → 재현 가능
    groups = {"train": [], "validation": [], "test": []}
    if test_bearings:
        groups["test"] = [p for p in paths if p.parent.name in test_bearings]
        paths = [p for p in paths if p.parent.name not in test_bearings]
        if {bearing_label(p.parent.name) for p in groups["test"]} != {0, 1}:
            raise ValueError("test 베어링에 정상과 고장이 모두 있어야 합니다 (한쪽만 있으면 오판율을 잴 수 없음)")
    for label in (0, 1):                 # 정상/고장 비율이 각 세트에 고르게 가도록 라벨별로 나눈다
        subset = [p for p in paths if bearing_label(p.parent.name) == label]
        if len(subset) < 2:
            raise ValueError(f"라벨 {label}의 파일이 너무 적습니다: {len(subset)}개")
        rng.shuffle(subset)
        if test_bearings:
            a = max(1, int(len(subset) * .85))
            parts = (subset[:a], subset[a:], [])
        else:
            a, b = int(len(subset) * .7), int(len(subset) * .15)
            parts = (subset[:a], subset[a:a + b], subset[a + b:])
        for name, selected in zip(groups, parts):
            groups[name].extend(selected)
    for name in groups:
        rng.shuffle(groups[name])
        groups[name] = [str(p.relative_to(ROOT)) for p in groups[name]]  # 루트 기준 상대 경로로 저장
    return groups


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--processed-dir", default="outputs/manual/processed")
    parser.add_argument("--output", default="outputs/manual/split_holdout.json")
    parser.add_argument("--test-bearings", nargs="+")
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

    paths = sorted((ROOT / args.processed_dir).rglob("*_windows.npy"))
    if not paths:
        raise SystemExit("전처리된 파일이 없습니다. 4장 스크립트를 먼저 실행하세요.")
    groups = create_split(paths, args.test_bearings, args.seed)

    # 같은 파일이 두 세트에 들어가면 안 된다 (데이터 누수)
    all_files = sum(groups.values(), [])
    assert len(all_files) == len(set(all_files)), "중복 파일 발견"

    out = ROOT / args.output
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(groups, indent=2) + "\n")
    for name, files in groups.items():
        per_bearing = Counter(Path(f).parent.name for f in files)
        print(f"{name:10s} 파일 {len(files):3d}개  {dict(sorted(per_bearing.items()))}")
    print(f"\n저장: {out.relative_to(ROOT)}")


if __name__ == "__main__":
    main()